# Build a channel report notebook from scratch

*Python Foundations · Lesson 8 final exercise*

The marketing team wants to know how much of January 2024's referral traffic came from Facebook and Instagram, and how much from every other site. Answer it from the full session export.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between this one and the **Check your work** cell.

1. Download `sessions_export_2024_01.csv` from the program's data store, [sessions_export_2024_01.csv](https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/sessions_export_2024_01.csv), and upload it to the `simmer-python` folder in your Drive.
2. Below this cell, add a text cell with a heading and a sentence that says what your notebook does.
3. Install `tldextract` in your first code cell.
4. Mount your Drive. Set `CSV_PATH` to the file's path, and read the file with `csv.DictReader` into `rows`.
5. Create `referral_rows`: a list of the rows whose `session_medium` is `referral`.
6. Define `site_name(hostname)`, which returns the site's name for a hostname, in lowercase, with `tldextract`, as in Topic 8.3.
7. Create `channel_totals`: a dictionary with two keys and their total sessions in `referral_rows`:
   - `"Organic Social"`: rows whose `session_source` has the site name `facebook` or `instagram`
   - `"Referral"`: every other row
8. Run the **Check your work** cell.
9. Choose **Runtime → Restart session and run all**, and check that you get the same result.

**Expected output**: `len(rows)` is `1711`, `len(referral_rows)` is `365`, and `channel_totals` is `{'Referral': 2655, 'Organic Social': 4640}`. The order of the two keys doesn't matter.

In [ ]:
# @title Check your work
# @markdown Run this cell last. Add your own cells above it. If you change anything above, run the cells again from the top first.
# This cell only reads your functions and variables. It doesn't change anything.
def check_your_work():
    import os
    import sys
    found = globals()
    results = []

    if "tldextract" in sys.modules:
        results.append((True, "tldextract is installed and imported."))
    else:
        results.append((False, "tldextract isn't imported. Install it with %pip install tldextract in your first code cell, then import it."))

    path = found.get("CSV_PATH")
    if not isinstance(path, str):
        results.append((False, "CSV_PATH isn't defined. Set it to the file's path in Drive, copied from the Files panel."))
    elif not path.startswith("/content/drive/"):
        results.append((False, f"CSV_PATH is {path!r}, which isn't in your Drive. Mount Drive, then copy the path of the file in drive/MyDrive/simmer-python from the Files panel. It starts with /content/drive/."))
    elif not path.endswith("sessions_export_2024_01.csv"):
        results.append((False, f"CSV_PATH is {path!r}. The exercise uses January's export, sessions_export_2024_01.csv."))
    elif not os.path.exists(path):
        results.append((False, f"There's no file at {path!r}. Check that Drive is mounted and that the file is in the simmer-python folder, then copy its path from the Files panel."))
    else:
        results.append((True, "CSV_PATH points to January's export in your Drive."))

    rows = found.get("rows")
    if not isinstance(rows, list) or not all(isinstance(row, dict) for row in rows):
        results.append((False, "rows should be a list of dictionaries, one per row of the file: list(csv.DictReader(file))."))
    elif len(rows) == 1711:
        results.append((True, "rows has all 1,711 rows of the export."))
    else:
        results.append((False, f"rows has {len(rows)} rows, but sessions_export_2024_01.csv has 1,711. Read January's file with csv.DictReader."))

    referral_rows = found.get("referral_rows")
    if not isinstance(referral_rows, list):
        results.append((False, "referral_rows should be a list of the rows whose session_medium is referral."))
    elif len(referral_rows) == 365 and all(row.get("session_medium") == "referral" for row in referral_rows):
        results.append((True, "referral_rows has the 365 referral rows."))
    elif len(referral_rows) == 1711:
        results.append((False, "referral_rows has every row of the export. Keep only the rows where row[\"session_medium\"] == \"referral\"."))
    elif len(referral_rows) == 0:
        results.append((False, "referral_rows is empty. The medium is written in lowercase in the file: row[\"session_medium\"] == \"referral\"."))
    else:
        results.append((False, f"referral_rows has {len(referral_rows)} rows, but 365 rows have the medium referral. Keep the rows where row[\"session_medium\"] == \"referral\"."))

    site_name = found.get("site_name")
    if not callable(site_name):
        results.append((False, "site_name() isn't defined. Run the cell that defines it."))
    else:
        cases = [("lm.facebook.com", "facebook"), ("l.instagram.com", "instagram"), ("www.pinterest.co.uk", "pinterest"), ("M.Facebook.com", "facebook")]
        wrong = []
        for hostname, expected in cases:
            try:
                got = site_name(hostname)
            except Exception as error:
                got = f"{type(error).__name__}: {error}"
            if got != expected:
                wrong.append((hostname, expected, got))
        if not wrong:
            results.append((True, "site_name() returns the site's name for every hostname."))
        else:
            hostname, expected, got = wrong[0]
            message = f"site_name({hostname!r}) should return {expected!r}, but it returned {got!r}."
            if got in ("co", "com"):
                message += " Use tldextract.extract(hostname).domain instead of splitting on dots."
            elif isinstance(got, str) and got.lower() == expected:
                message += " Return the name in lowercase."
            elif isinstance(got, str) and got.lower().startswith(expected + "."):
                message += " That's the name with its suffix. Use .domain."
            results.append((False, message))

    totals = found.get("channel_totals")
    expected = {"Organic Social": 4640, "Referral": 2655}
    if totals is None:
        results.append((False, "channel_totals isn't defined. Run the cell that creates it."))
    elif not isinstance(totals, dict):
        results.append((False, f"channel_totals should be a dictionary with Organic Social and Referral, but it is {totals!r}."))
    elif totals == expected:
        results.append((True, "channel_totals has the right sessions for both channels."))
    elif not totals:
        results.append((False, "channel_totals is empty. If its cell stopped with an error, read the error, fix it and run the cell again."))
    elif totals == {"Organic Social": 199, "Referral": 166}:
        results.append((False, "channel_totals counts rows, not sessions. Add int(row[\"sessions\"]) to the channel's total for every row."))
    elif sum(v for v in totals.values() if isinstance(v, int)) > 7295:
        results.append((False, f"channel_totals has more sessions than the referral rows (7,295). Total referral_rows, not rows. It is {totals!r}."))
    elif totals.get("Organic Social", 0) == 0:
        results.append((False, "channel_totals has no Organic Social sessions. Compare the site name with \"facebook\" and \"instagram\": site_name(row[\"session_source\"]) in {\"facebook\", \"instagram\"}."))
    elif totals.get("Organic Social") == 2677:
        results.append((False, "Organic Social only has the rows from facebook.com and instagram.com. Sources such as m.facebook.com and l.instagram.com are Facebook and Instagram too: compare their site_name()."))
    elif sum(v for v in totals.values() if isinstance(v, int)) < 7295:
        results.append((False, f"channel_totals adds up to fewer than the 7,295 sessions in referral_rows. Add each row's sessions to its channel's total with channel_totals.get(channel, 0) + int(row[\"sessions\"]), instead of replacing it. It is {totals!r}."))
    else:
        results.append((False, f"channel_totals should be {expected}, but it is {totals!r}."))

    for passed, message in results:
        print(("✅ " if passed else "❌ ") + message)
    if all(passed for passed, message in results):
        print("\nEverything matches. Now restart the session and run all, and check that you get the same result.")
    else:
        print("\nFix the first ❌, run the cells above again from the top, then run this cell again.")


check_your_work()

<details>
<summary><b>Show hint</b></summary>

Build the notebook in the order the brief lists the steps, and run each cell before you write the next one. Check `len(rows)` and `len(referral_rows)` before you total anything. `site_name()` is the same function as in Topic 8.3. For the totals, the channel is `"Organic Social"` if `site_name(row["session_source"])` is in `{"facebook", "instagram"}`, and `"Referral"` if it isn't. Add `int(row["sessions"])` to the channel's total with `.get(channel, 0)`, as in Topic 8.1.

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
%pip install tldextract
```

```python
from google.colab import drive

drive.mount("/content/drive")
```

```python
import csv

import tldextract

CSV_PATH = "/content/drive/MyDrive/simmer-python/sessions_export_2024_01.csv"

with open(CSV_PATH, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
len(rows)
```

```python
referral_rows = [row for row in rows if row["session_medium"] == "referral"]
len(referral_rows)
```

```python
def site_name(hostname):
    return tldextract.extract(hostname).domain.lower()
```

```python
channel_totals = {}
for row in referral_rows:
    if site_name(row["session_source"]) in {"facebook", "instagram"}:
        channel = "Organic Social"
    else:
        channel = "Referral"
    channel_totals[channel] = channel_totals.get(channel, 0) + int(row["sessions"])
channel_totals
```

</details>